# Phase 4 — Training on Shakespeare

> **Historical — negative result.** This run scored **test BLEU 15.07 against a 19.22 copy baseline**: worse than echoing the input unchanged. It uses the word-level vocabulary, superseded by SentencePiece in Phase 5.
>
> It is kept because it is the measurement that justifies pretraining. Full analysis: [`docs/phase4_findings.md`](../docs/phase4_findings.md).

Phase 2 proved the model learns; Phase 3 built the data. This trains it for real and asks the only question that matters: **does it beat the copy baseline?**

(It does not. That is the finding.)

In [ ]:
import os
import sys

for candidate in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(candidate, "src")):
        ROOT = candidate
        sys.path.insert(0, ROOT)
        break

import matplotlib.pyplot as plt
import torch

from src.config import TransformerConfig, get_device
from src.dataset import TranslationDataset, load_split, make_dataloader
from src.evaluate import copy_baseline, bleu
from src.inference import translate_corpus
from src.train import fit, load_checkpoint, overfit_batch
from src.transformer import build_model
from src.vocab import Vocab

%matplotlib inline
DATA = os.path.join(ROOT, "data")
CHECKPOINT = os.path.join(ROOT, "checkpoints", "best.pt")

device = get_device()
torch.manual_seed(0)

train_src, train_tgt = load_split(DATA, "train")
valid_src, valid_tgt = load_split(DATA, "valid")
test_src, test_tgt = load_split(DATA, "test")
vocab = Vocab.load(os.path.join(DATA, "vocab.json"))

train_loader = make_dataloader(TranslationDataset(train_src, train_tgt, vocab), batch_size=64)
valid_loader = make_dataloader(TranslationDataset(valid_src, valid_tgt, vocab),
                               batch_size=64, shuffle=False)

valid_baseline = copy_baseline(valid_src, valid_tgt)
test_baseline = copy_baseline(test_src, test_tgt)

print("device:", device, " vocab:", len(vocab))
print("copy baseline   valid %.2f   test %.2f" % (valid_baseline, test_baseline))

## 1. Overfit one real batch

Same gate as the toy task, now on real sentences. If the model cannot memorise 32 Shakespeare pairs, something in the data pipeline is wrong and a long run would only waste time finding out.

Dropout off, so nothing puts a floor under the loss.

In [ ]:
probe_cfg = TransformerConfig(vocab_size=len(vocab), d_model=256, num_heads=4,
                              num_layers=3, d_ff=1024, dropout=0.0, max_len=200)
probe = build_model(probe_cfg).to(device)

batch = tuple(t.to(device) for t in next(iter(make_dataloader(
    TranslationDataset(train_src, train_tgt, vocab), batch_size=32))))

history = overfit_batch(probe, batch, steps=300, log_every=100)
print("\nfinal loss %.4f   accuracy %.1f%%" % (history["loss"][-1], 100 * history["acc"][-1]))

## 2. The training run

Three choices worth stating:

**Model size.** 3 layers, d_model 256 — smaller than the paper's 512/8/6, because 18k sentence pairs cannot support a paper-sized model without memorising them.

**Dropout 0.2.** A pilot at 0.3 showed train and validation loss almost identical, which is underfitting, not overfitting. Regularisation was costing more than it bought.

**Selection on BLEU, not validation loss.** They diverge — loss keeps improving on token-level confidence after translation quality has peaked. The checkpoint kept is the best-BLEU one, and training stops when it stalls.

In [ ]:
torch.manual_seed(0)

config = TransformerConfig(vocab_size=len(vocab), d_model=256, num_heads=4,
                           num_layers=3, d_ff=1024, dropout=0.2, max_len=200)
model = build_model(config).to(device)

print("parameters:", f"{model.count_parameters()['TOTAL']:,}")

In [ ]:
history = fit(
    model, config, train_loader, valid_loader,
    valid_src, valid_tgt, vocab, device,
    epochs=60, patience=8, lr=1e-3, warmup=400,
    checkpoint_path=CHECKPOINT, baseline=valid_baseline,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["valid_loss"], label="valid")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["epoch"], history["bleu"], color="tab:green", label="model")
axes[1].axhline(valid_baseline, color="tab:red", ls="--", label="copy baseline")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("BLEU")
axes[1].set_title("Validation BLEU"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Test set

*Romeo and Juliet* — a play the model has never seen. The checkpoint reloaded here is the best-BLEU one, not the final epoch.

Decoding is greedy. Beam search was added later and measured at +1.4 BLEU with length penalty alpha 1.5 (not the textbook 0.6) - see the findings doc.

In [ ]:
best, meta = load_checkpoint(CHECKPOINT, device=device)
print("loaded epoch %d, valid BLEU %.2f" % (meta["epoch"], meta["score"]))

predictions = translate_corpus(best, test_src, vocab, device)
test_bleu = bleu(predictions, test_tgt)

print()
print("test BLEU      %6.2f" % test_bleu)
print("copy baseline  %6.2f" % test_baseline)
print("gain           %+6.2f" % (test_bleu - test_baseline))

In [ ]:
for i in (3, 11, 25, 60):
    print("SHAKESPEARE:", test_src[i])
    print("MODEL      :", predictions[i])
    print("REFERENCE  :", test_tgt[i])
    print()

## Result

**Test BLEU 15.07 — below the 19.22 copy baseline.** The architecture is correct and the pipeline is sound; there is simply not enough data. 474,478 words against 8.1M parameters is 0.06 words per parameter where the rule of thumb is ~20 — a **340× shortfall**.

Diagnostics, follow-up experiments and the reasoning are in [`docs/phase4_findings.md`](../docs/phase4_findings.md).

Next: Phases 5–8 — learn English from 200M words of Gutenberg first, so these 18k pairs only have to teach the style mapping.